In [116]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import io
import numpy as np
import re
import matplotlib.pyplot as plt
from time import sleep

In [117]:
# PBOC statistics front page

def get_html(url):
    sleep(0.5)
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
    r=requests.get(url, headers=headers)
    return r

In [138]:
def get_year_data_cat_list(url):
    domain='http://www.pbc.gov.cn'
    r=get_html(url)
    soup=BeautifulSoup(r.content, 'html.parser')
    data_link={i.get_text(): domain+i.get('href') for i in 
            soup.find('div', {'name': '右侧内容'}).find_all('a')}
    return data_link

In [119]:
def get_data_list(url):
    r=get_html(url)
    domain='http://www.pbc.gov.cn'
    soup=BeautifulSoup(r.content, 'html.parser')

    data={}
    for table in soup.find_all('table', {'class': 'a2015'}):
        lst=[]
        for i, item in enumerate(table.find_all('td')):
            if i ==0:
                title=item.get_text(strip=True)
            else:
                if item.find('a') is None:
                    lst.append({item.get_text(): None})
                else:
                    link=item.find('a').get('href')
                    if link is None:
                        lst.append({item.get_text(): None})
                    else:
                        lst.append({item.get_text(): domain+link})
        data[title]=lst
    return data

In [120]:
# load PBOC data list (output: result)

url='http://www.pbc.gov.cn/diaochatongjisi/116219/index.html'
r=get_html(url)
soup=BeautifulSoup(r.content, 'html.parser')
domain='http://www.pbc.gov.cn'
data_link_by_year={re.search(r'[0-9]{4}', i.get_text())[0]: domain+i.get('href') for i in 
                   soup.find('table', {'id': '11854'}).find_next_sibling('table').find_all('a')}

result=[]
for key, item in data_link_by_year.items():
    dict_data={}
    dict_data['year']=key
    dict_data['cat_link']=get_year_data_cat_list(item)
    result.append(dict_data)


for i, year in enumerate(result):
    lst_result=[]
    for key, link in year['cat_link'].items():
        dict_data={}
        dict_data['name']=key
        dict_data['result']=get_data_list(link)
        lst_result.append(dict_data)

    result[i]['cat_result']=lst_result

In [167]:
# 社会融资规模存量统计表Aggregate Financing to the Real Economy (Stock)

def get_table_link(table, file_format):
    output={}
    for year in result:
        for ind in year['cat_result']:
                for key, value in ind['result'].items():
                        if table in key:
                            for i in value:
                                if i.get(file_format) is not None:
                                    output[year['year']]=i.get(file_format)

    return output

def get_table_link(table, file_format):
    return {
        year['year']: next(
            (i[file_format] for ind in year['cat_result']
             for key, value in ind['result'].items()
             if table in key
             for i in value
             if file_format in i),
            None
        )
        for year in result
        if any(table in key and any(file_format in i for i in value)
               for ind in year['cat_result']
               for key, value in ind['result'].items())
    }


In [168]:
def format_social_finance_df(df):
    df=df.dropna(how='all', axis=1)
    col=df.iloc[5].str.strip()
    col[0]='月份'
    df.columns=col
    df=df[df['月份'].astype(str).str.contains(r'[0-9]{4}\.[0-9]{2}', regex=True)]
    df=df.set_index('月份')
    df.index=pd.to_datetime(df.index, format='%Y.%m')
    df=df.apply(pd.to_numeric)
    return df

In [169]:
def format_money_supply(df):
    df=df[df[3].astype(str).str.contains(r'\.[0-9]{2}$', regex=True)].dropna(subset=[2])
    df=df.set_index(2)
    df.columns=df.loc['项目 Item']
    df=df[df.columns[df.columns.astype(str).str.contains(r'\.[0-9]{2}$', regex=True)]]
    df=df.drop('项目 Item')
    df=df.transpose()
    df.index=pd.to_datetime(df.index, format='%Y.%m')
    df=df.apply(pd.to_numeric)
    return df

In [216]:
def format_reserve(df):
    
    df=df.set_index(0)
    df=df.dropna(subset=[1], axis=1)
    df=df.dropna(subset=[1])
    df=df.transpose()
    col=['Official reserve assets', 'Date', 'Unit', 'Foreign currency reserves', 'IMF reserve position', 'SDRs', 'Gold', 
         'Gold(0,000 Ounce)', 'Other reserve assets', 'Total', 'Remarks']
    df.columns=col
    df['Gold(0,000 Ounce)']=df['Gold(0,000 Ounce)'].astype(str).str.replace('万盎司', '')
    df=df[df['Unit'].astype(str).str.contains('USD')]
    df=df.set_index(['Date'])
    df=df.drop(['Official reserve assets', 'Unit', 'Remarks'], axis=1)
    df.index=pd.to_datetime(df.index, format='%Y.%m')
    df=df.apply(lambda x: x.astype(float))
#     df=df.apply(pd.to_numeric)
    return df

In [171]:
def load_data(table_name, func):
    file_format='htm'
    links=get_table_link(table_name, file_format)

    dfs=[] 
    for key, url in links.items():
        data=get_html(url)
        df=pd.read_html(data.content)[0]
        df=func(df)
        dfs.append(df)
    return dfs

In [195]:

table_name_func={'货币供应量': format_money_supply, 
                 '社会融资规模增量统计表': format_social_finance_df,
                 '官方储备资产': format_reserve}

# table_name='社会融资规模增量统计表'
table_name='货币供应量'
table_name='官方储备资产'
# file_format='htm'
# links=get_table_link(table_name, file_format)
url=links['2015']
data=get_html(url)

In [232]:

df=pd.read_html(data.content)[0]

# df=df.dropna(subset=[1])
# df=df.dropna(subset=[0], axis=1)
# df=df.set_index(0)
# df=df.dropna(subset=[1])
df=df.transpose()
col=['Official reserve assets', 'Date', 'Unit', 'Foreign currency reserves', 'IMF reserve position', 'SDRs', 'Gold', 
        'Gold(0,000 Ounce)', 'Other reserve assets', 'Total', 'Remarks']
df.columns=col
# df['Gold(0,000 Ounce)']=df['Gold(0,000 Ounce)'].astype(str).str.replace('万盎司', '')
# df=df[df['Unit'].astype(str).str.contains('USD')]
# df=df.set_index(['Date'])
# df=df.drop(['Official reserve assets', 'Unit', 'Remarks'], axis=1)
# df.index=pd.to_datetime(df.index, format='%Y.%m')
# df=df.apply(lambda x: x.astype(float))

df
# format_reserve(df)

,Official reserve assets,Date,Unit,Foreign currency reserves,IMF reserve position,SDRs,Gold,"Gold(0,000 Ounce)",Other reserve assets,Total,Remarks
0,官方储备资产 Official reserve assets,NaN,项目 Item,NaN,1.外汇储备 Foreign currency reserves,2.基金组织储备头寸 IMF reserve position,3.特别提款权 SDRs,4.黄金 Gold,NaN,5.其他储备资产 Other reserve assets,合计 Total
1,官方储备资产 Official reserve assets,NaN,2015.06,NaN,36938.38,45.67,105.45,623.97,（5332万盎司）,0.00,37713.47
2,官方储备资产 Official reserve assets,NaN,2015.07,NaN,36513.10,43.73,104.60,592.38,（5393万盎司）,0.68,37254.48
3,官方储备资产 Official reserve assets,NaN,2015.08,NaN,35573.81,47.27,105.29,617.95,（5445万盎司）,-2.48,36341.84
4,官方储备资产 Official reserve assets,NaN,2015.09,NaN,35141.20,46.89,104.68,611.89,（5493万盎司）,-1.96,35902.70
5,官方储备资产 Official reserve assets,NaN,2015.1,NaN,35255.07,46.38,103.61,632.61,（5538万盎司）,2.63,36040.29
6,官方储备资产 Official reserve assets,单位：亿美元 US$ 100million,2015.11,NaN,34382.84,45.96,101.79,595.22,（5605万盎司）,3.96,35129.78
7,官方储备资产 Official reserve assets,单位：亿美元 US$ 100million,2015.12,NaN,33303.62,45.47,102.84,601.91,（5666万盎司）,7.27,34061.11
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [197]:
dfs=load_data(table_name, table_name_func[table_name])
df=pd.concat(dfs)
df=df.apply(pd.to_numeric)
df.index=pd.to_datetime(df.index, format='%Y.%m')
df=df.sort_index()
df
# df[['货币和准货币（M2）', '货币（M1）']].plot(label=['M2', 'M1'])

ValueError: Length mismatch: Expected axis has 9 elements, new values have 11 elements

In [30]:
# '货币统计概览 Money and Banking Statistics'
list(filter(lambda x: '货币统计概览' in x['name'], [i for i in result[0]['cat_result']]))
[j if isinstance(j, dict) else None for i in result[0]['cat_result'] for _, j in i.items()]

[None,
 {'社会融资规模增量统计表Aggregate Financing to the Real Economy (Flow)': [{'htm': 'http://www.pbc.gov.cn/diaochatongjisi/resource/cms/2024/08/2024081416043961436.htm'},
   {'xls': 'http://www.pbc.gov.cn/diaochatongjisi/resource/cms/2024/08/2024081417423516340.xlsx'},
   {'pdf': 'http://www.pbc.gov.cn/diaochatongjisi/resource/cms/2024/08/2024081417423566946.pdf'}],
  '社会融资规模存量统计表Aggregate Financing to the Real Economy (Stock)': [{'htm': 'http://www.pbc.gov.cn/diaochatongjisi/resource/cms/2024/08/2024081416052616168.htm'},
   {'xls': 'http://www.pbc.gov.cn/diaochatongjisi/resource/cms/2024/08/2024081417423514097.xlsx'},
   {'pdf': 'http://www.pbc.gov.cn/diaochatongjisi/resource/cms/2024/08/2024081417423488210.pdf'}],
  '地区社会融资规模增量统计表Aggregate Financing to the Real Economy（flow）by Province': [{'Q1': 'http://www.pbc.gov.cn/diaochatongjisi/116219/116225/5344502/index.html'},
   {'Q2': 'http://www.pbc.gov.cn/diaochatongjisi/116219/116225/5419191/index.html'},
   {'Q3': None},
   {'Q4': None}]},